# Ammar: C1 ViT from scratch + C2 PEFT on a Colab T4

Kernel: **Colab, T4 GPU** (Runtime → Change runtime type → T4). Run the cells top to bottom.

Re-runs the tuning stages on seeds 0–4 and then the chosen `best` / `fast` configs on the report seeds
10–19, all tagged `hw=colab-t4`, so the times in the report come from the same GPU type as the rest of
the team.

**Resumable:** rows are logged to a copy of `results/ammar.csv` on your Google Drive, so they survive a
disconnect. Just run all cells again: rows already logged for `hw=colab-t4` are skipped.

**Nothing is pushed from here.** At the end, the merged `ammar.csv` and the figures are downloaded to your
computer; you copy them into the repo and commit them yourself.

Before running: the branch below must already be pushed to GitHub (Colab clones it from there).
If you change the code, **restart the kernel** before running again.

In [ ]:
# 1. Get the code (read-only: a token with Contents: Read is enough)
import os
import shutil
from getpass import getpass

REPO = "/content/repo"
BRANCH = "ammar/vit-scratch"     # the branch with your code

token = getpass("GitHub token (read access is enough): ")
url = f"https://{token}@github.com/MassPaol/comp433-final-project.git"
del token
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
else:
    !git -C {REPO} pull -q
del url

%cd {REPO}
!pip install -q -e . -r requirements.txt

In [ ]:
# 2. Mount Google Drive (results copy lives here so a disconnect loses nothing)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Data: copy CIFAR-10 from the team folder if it's there, otherwise download it
DRIVE_DATA = "/content/drive/MyDrive/comp433_cache/data"   # needs the comp433_cache shortcut in My Drive
LOCAL_DATA = "/content/data"
os.makedirs(LOCAL_DATA, exist_ok=True)
src = f"{DRIVE_DATA}/cifar-10-python.tar.gz"
if os.path.exists(src) and not os.path.exists(f"{LOCAL_DATA}/cifar-10-python.tar.gz"):
    shutil.copy(src, LOCAL_DATA)
os.environ["COMP433_DATA"] = LOCAL_DATA

In [ ]:
# 4. Import the project code (after COMP433_DATA is set), check the GPU and the data
import torch
from common import data

assert data.DATA_ROOT == os.environ["COMP433_DATA"], "Restart the kernel and run all cells again."
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(), (
    f"Expected a T4 GPU, got {torch.cuda.get_device_name() if torch.cuda.is_available() else 'no GPU'}")
HW = "colab-t4"
X, y, Xv, yv, classes = data.load_instance(0)
print(torch.cuda.get_device_name(), X.shape, Xv.shape, [data.CLASS_NAMES[c] for c in classes])

In [ ]:
# 5. Log to a copy of results/ammar.csv on Drive
RESULTS_DIR = "/content/drive/MyDrive/comp433_ammar_results"
RESULTS = f"{RESULTS_DIR}/ammar.csv"
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists(RESULTS):
    shutil.copy("results/ammar.csv", RESULTS)
print("logging to", RESULTS)

In [ ]:
# 6. C1 ViT from scratch: tuning stages on seeds 0-4
for stage in ["s0", "s1", "s2"]:
    !python -W ignore -m c1.vit_scratch.run --stage {stage} --seeds tune --hw {HW} --results-dir {RESULTS_DIR}

In [ ]:
# 7. C2 PEFT: tuning stages on seeds 0-4
for stage in ["p0", "p1"]:
    !python -W ignore -m c2.peft.run --stage {stage} --seeds tune --hw {HW} --results-dir {RESULTS_DIR}

In [ ]:
# 8. Final configs on the report seeds 10-19.
# Fill these in once best/fast are chosen from the tuning results (NOTES.md). Leave empty to skip.
C1_FINAL = []    # e.g. ["s2_spt", "s1_dim64"]
C2_FINAL = []    # e.g. ["bitfit_lr1e-3", "head"]
if C1_FINAL:
    !python -W ignore -m c1.vit_scratch.run --config {" ".join(C1_FINAL)} --seeds report --hw {HW} --results-dir {RESULTS_DIR}
if C2_FINAL:
    !python -W ignore -m c2.peft.run --config {" ".join(C2_FINAL)} --seeds report --hw {HW} --results-dir {RESULTS_DIR}

In [ ]:
# 9. Figures from the T4 rows
!python -m c1.vit_scratch.figures --hw {HW} --results {RESULTS}
!python -m c2.peft.figures --hw {HW} --results {RESULTS}

In [ ]:
# 10. Download the results and figures to your computer (nothing is pushed)
from google.colab import files
!cd {REPO} && zip -q -j /content/ammar_t4.zip {RESULTS} report/figures/c1_vit_* report/figures/c2_peft_*
files.download("/content/ammar_t4.zip")

### After downloading

Unzip, then on your laptop in the repo folder (project Python environment active):

```
python -c "from c1.vit_scratch.resume import merge_results; print(merge_results(r'PATH\TO\ammar.csv', 'results/ammar.csv'), 'rows added')"
```

Copy the `c1_vit_*` / `c2_peft_*` figure files into `report/figures/`, then commit when you're ready.

**Before committing this notebook, clear its outputs** (Edit → Clear All Outputs): the repo is public.